# Full Sweep Demo

Demonstrates the complete measurement workflow described in the top-level
CLAUDE.md: configure all four instruments, sweep the Avtech pulse voltage
over a user-defined range while recording (voltage, current, lock-in
signal) via `measurement.sweep.run_voltage_sweep`, save the result with
`measurement.data_io`, and plot I-V and optical-intensity-vs-voltage
curves with `measurement.plotting`.

**No real instruments are connected to this development machine.** If any
instrument fails to connect, this notebook falls back to clearly-labeled
synthetic `dummy_data` (built to loosely resemble a laser-like I-V/optical
threshold curve) so the saving and plotting logic can still be
demonstrated end-to-end. On the lab computer, with all four instruments
connected, the same cells should run the real sweep instead.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from instruments.qdac import QDac
from instruments.avtech import Avtech
from instruments.rigol import Rigol
from instruments.mfli import MFLI
from instruments import config
from measurement.sweep import run_voltage_sweep, SweepConfig
from measurement import data_io, plotting

## Instantiate and configure all four instruments

In [ ]:
qdac = QDac()
avtech = Avtech()
rigol = Rigol()
mfli = MFLI()

instruments_connected = False

try:
    # QDac: trigger/gate signal generator that synchronizes the Avtech
    # pulse and the MFLI lock-in reference (see instruments/qdac.py).
    qdac.connect()
    qdac.channel_init(1)
    qdac.channel_init(2)
    gate_voltage = 5.0
    qdac.configure_square_wave(
        1,
        frequency=200,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source="INT1",
    )
    qdac.configure_square_wave(
        2,
        frequency=2000,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source="INT1",
        delay=0.0125e-3,
        duty_cycle=50,
    )
    qdac.start_square_wave(1)
    qdac.start_square_wave(2)
    qdac.fire_internal_trigger(1)

    # Avtech: swept pulse voltage source, externally triggered by the QDac.
    avtech.connect()
    avtech.remote()
    avtech.set_trigger("EXT")
    avtech.set_pulse_width(80e-6)
    avtech.output_on()

    # Rigol: reads back DUT voltage/current waveforms for each pulse.
    rigol.connect()
    rigol.configure_timebase(1e-5, offset=50e-6)
    rigol.configure_channel(config.DEFAULT_CHANNEL_ROLES["voltage"], 2.0)
    rigol.configure_channel(config.DEFAULT_CHANNEL_ROLES["current"], 2.0)
    rigol.configure_channel(config.DEFAULT_CHANNEL_ROLES["trigger"], 2.0)
    rigol.configure_trigger(
        config.DEFAULT_CHANNEL_ROLES["trigger"], level=2.0, slope="POSITIVE"
    )

    # MFLI: reads the lock-in signal corresponding to optical intensity.
    mfli.connect()
    mfli.apply_default_configuration()

    instruments_connected = True
    print("All four instruments connected and configured.")
except Exception as exc:
    print(f"Hardware not available: {exc}")
    print("Falling back to synthetic dummy_data later in this notebook.")

## Define the voltage sweep range

In [ ]:
voltages = np.linspace(2.0, 10.0, 5)
print("Sweep voltages (V):", voltages)

## Run the sweep, or fall back to synthetic dummy_data

In [ ]:
df = None

if instruments_connected:
    sweep_config = SweepConfig(
        voltage_channel=config.DEFAULT_CHANNEL_ROLES["voltage"],
        current_channel=config.DEFAULT_CHANNEL_ROLES["current"],
        idle_voltage=5.0,
    )
    try:
        df = run_voltage_sweep(
            avtech, rigol, mfli, qdac, voltages=voltages, sweep_config=sweep_config
        )
        print("Sweep completed using real hardware.")
    except Exception as exc:
        print(f"Sweep failed mid-run: {exc}")

used_synthetic_data = df is None
if used_synthetic_data:
    # NOTE: hardware not available (or the sweep failed) -- this is
    # clearly-labeled synthetic dummy data, NOT a real measurement. It
    # loosely mimics a laser-like I-V/optical threshold: negligible
    # current/intensity below ~2 V, roughly linear current and intensity
    # above threshold.
    rng = np.random.default_rng(0)
    threshold_voltage = 2.0
    dut_voltage = 0.9 * voltages + rng.normal(0, 0.02, size=voltages.shape)
    above_threshold = np.clip(voltages - threshold_voltage, 0, None)
    dut_current = above_threshold * 0.01 + rng.normal(0, 0.0005, size=voltages.shape)
    lockin_x = above_threshold * 0.005 + rng.normal(0, 0.0002, size=voltages.shape)
    lockin_y = rng.normal(0, 0.0002, size=voltages.shape)
    lockin_r = np.hypot(lockin_x, lockin_y)
    lockin_phase = np.arctan2(lockin_y, lockin_x)

    dummy_data = pd.DataFrame(
        {
            "set_voltage": voltages,
            "dut_voltage": dut_voltage,
            "dut_current": dut_current,
            "lockin_x": lockin_x,
            "lockin_y": lockin_y,
            "lockin_r": lockin_r,
            "lockin_phase": lockin_phase,
        }
    )
    print("Using synthetic dummy_data (NOT a real measurement).")
    df = dummy_data

df

## Save the sweep result

In [ ]:
output_dir = project_root / "data"
csv_path = output_dir / "sweep_demo.csv"
data_io.save_csv(df, csv_path)
print(f"Saved CSV to {csv_path}")

try:
    hdf5_path = output_dir / "sweep_demo.h5"
    data_io.save_hdf5(df, hdf5_path)
    print(f"Saved HDF5 to {hdf5_path}")
except ImportError as exc:
    print(f"Skipping HDF5 save: {exc}")

## Reload the CSV to confirm the round-trip

In [ ]:
reloaded = data_io.load_csv(csv_path)
reloaded

## Plot I-V and optical-intensity-vs-voltage curves

In [ ]:
if used_synthetic_data:
    print("NOTE: the plots below are based on synthetic dummy_data, not a real measurement.")
else:
    print("The plots below are based on a real sweep measurement.")

In [ ]:
plotting.plot_iv_curve(df, show=True)

In [ ]:
plotting.plot_intensity_curve(df, show=True)

## Clean up and disconnect all instruments

In [ ]:
try:
    if instruments_connected:
        avtech.output_off()
        qdac.abort_square_wave(1)
        qdac.abort_square_wave(2)
        qdac.set_channel_voltage(1, 0.0)
        qdac.set_channel_voltage(2, 0.0)
        print("Avtech output disabled; QDac trigger/gate channels aborted and zeroed.")
except Exception as exc:
    print(f"Error during instrument cleanup: {exc}")

In [ ]:
for name, instrument in [("QDac", qdac), ("Avtech", avtech), ("Rigol", rigol), ("MFLI", mfli)]:
    try:
        if instruments_connected:
            instrument.disconnect()
            print(f"Disconnected from {name}.")
    except Exception as exc:
        print(f"Error disconnecting {name}: {exc}")

if not instruments_connected:
    print("Instruments were never connected; nothing to disconnect.")